# TP 06 — Seq2Seq with Bahdanau Attention

**Course:** Natural Language Processing  
**Session:** 6 / 8

---

## Context

In S05 the LSTM produced a single hidden state per sentence.
Today you build a **seq2seq model with Bahdanau attention** for EN→FR translation.
The decoder reads all encoder hidden states at each step, dynamically focusing on
the most relevant source tokens.

**Corpus design:** a systematic synthetic corpus of sentences of the form
*'the black cat sleeps'* → *'le chat noir dort'* (500 pairs, 10 colors × 10 animals × 10 verbs).
Because adjective order differs between EN and FR, the attention map must show
an off-diagonal pattern — directly visible evidence of what the model has learned.

> **Stack note:** this TP uses Keras 3 (ships with TensorFlow on Colab).
> The model is a **subclassed `keras.Model`** with a **bidirectional encoder**.
> All tensor ops use `keras.ops` (not `tf.*`) for Keras-3 compatibility.

---

## Roadmap

| Part | Task | New concept |
|------|------|-------------|
| 1 | Build corpus, vocabulary, encoding | SOS/EOS, padding |
| 2 | `BahdanauAttention` layer | custom `keras.Layer`, `keras.ops` |
| 3 | `Seq2SeqAttention` model | subclassed `keras.Model`, bidirectional encoder |
| 4 | Train | `model.fit` with teacher forcing |
| 5 | Greedy decoding | inference loop |
| 6 | Attention maps | heatmap — verify adjective reordering |
| 7 (bonus) | Beam search | width-k decoding |

---


## Setup


In [ ]:
# !pip install keras tensorflow scikit-learn matplotlib -q

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
import random

import keras
from keras import ops
from keras.layers import Embedding, LSTM, Dense, Layer, Bidirectional

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

EMB_DIM    = 64
LSTM_UNITS = 128   # decoder LSTM units; encoder uses LSTM_UNITS//2 per direction
ATTN_DIM   = 64
EPOCHS     = 100
BATCH_SIZE = 32
PAD, SOS, EOS, UNK = 0, 1, 2, 3

print(f'Keras: {keras.__version__}')
print('Setup OK.')


---
## Part 1 — Corpus, Vocabulary, Encoding


In [ ]:

# ── Corpus generation ─────────────────────────────────────────────────────────
# Synthetic corpus: "the {color} {animal} {verb}" → "le {animal_fr} {color_fr} {verb_fr}"
# 10 colors × 10 animals × 10 verbs = 1000 unique combinations; we use 500.
#
# This design is intentional:
#   1. Adjective order differs (EN: color before noun  VS  FR: noun before color)
#      → the attention map MUST show an off-diagonal pattern to translate correctly
#   2. 500 pairs prevents full memorisation → the model must generalise
#   3. Fixed vocabulary (~25 words per language) keeps training fast (~3 min on CPU)

COLORS_EN = ["black","white","red","blue","green","yellow","small","big","old","young"]
COLORS_FR = ["noir",  "blanc","rouge","bleu","vert","jaune","petit","grand","vieux","jeune"]
ANIMALS_EN = ["cat",  "dog",  "bird",  "fish",   "rabbit","horse","cow",  "lion", "duck", "bear"]
ANIMALS_FR = ["chat","chien","oiseau","poisson","lapin", "cheval","vache","lion","canard","ours"]
VERBS_EN   = ["sleeps","runs","eats","sings","flies","swims","jumps","drinks","walks","plays"]
VERBS_FR   = ["dort", "court","mange","chante","vole","nage","saute","boit", "marche","joue"]

import random
random.seed(42)

CORPUS = []
for ce, cf in zip(COLORS_EN, COLORS_FR):
    for ae, af in zip(ANIMALS_EN, ANIMALS_FR):
        for ve, vf in zip(VERBS_EN, VERBS_FR):
            # EN: the {color} {animal} {verb}
            # FR: le {animal_fr} {color_fr} {verb_fr}   ← adjective AFTER noun
            CORPUS.append((f"the {ce} {ae} {ve}", f"le {af} {cf} {vf}"))

random.shuffle(CORPUS)
CORPUS = CORPUS[:500]

src_sents = [en for en, fr in CORPUS]
tgt_sents = [fr for en, fr in CORPUS]

print(f'Corpus: {len(CORPUS)} pairs')
print('Sample (notice adjective order EN vs FR):')
for en, fr in CORPUS[:6]:
    print(f'  EN: {en}')
    print(f'  FR: {fr}')
    print()
print('Key insight:')
print('  EN: the BLACK cat sleeps  (adj before noun)')
print('  FR: le chat NOIR dort     (adj after noun)')
print('  The attention map MUST show this crossing to translate correctly.')


### 1.1 — Implement `build_vocab`


In [ ]:
def build_vocab(sentences: list[str]) -> tuple[dict, dict]:
    """
    Build word-to-index and index-to-word vocabularies.

    Reserves indices 0–3 for special tokens:
        0 = <PAD>,  1 = <SOS>,  2 = <EOS>,  3 = <UNK>
    Remaining words are assigned from index 4, ordered by frequency.

    Parameters
    ----------
    sentences : list of str

    Returns
    -------
    word2idx : dict
    idx2word : dict
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
src_w2i, src_i2w = build_vocab(src_sents)
tgt_w2i, tgt_i2w = build_vocab(tgt_sents)

SRC_VOCAB = len(src_w2i)
TGT_VOCAB = len(tgt_w2i)

print(f'Source vocab: {SRC_VOCAB}   Target vocab: {TGT_VOCAB}')
assert src_w2i['<PAD>'] == 0
assert src_w2i['<SOS>'] == 1
assert src_w2i['<EOS>'] == 2
print('Special token indices OK.')


### 1.2 — Implement `encode_sentences`


In [ ]:
def encode_sentences(
    sentences: list[str],
    word2idx: dict,
    max_len: int,
) -> np.ndarray:
    """
    Encode sentences to a padded integer matrix.

    Each sentence is wrapped with SOS (1) and EOS (2).
    Unknown words map to UNK (3).
    Post-padded with PAD (0) to max_len.

    Parameters
    ----------
    sentences : list of str
    word2idx  : dict
    max_len   : int

    Returns
    -------
    np.ndarray, shape (len(sentences), max_len), dtype int32
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
SRC_MAX = max(len(s.split()) for s in src_sents) + 2
TGT_MAX = max(len(t.split()) for t in tgt_sents) + 2

src_enc = encode_sentences(src_sents, src_w2i, SRC_MAX)
tgt_enc = encode_sentences(tgt_sents, tgt_w2i, TGT_MAX)

src_tr, src_va, tgt_tr, tgt_va = train_test_split(
    src_enc, tgt_enc, test_size=0.1, random_state=RANDOM_STATE
)

print(f'src: {src_enc.shape}  tgt: {tgt_enc.shape}')
print(f'Train: {len(src_tr)}   Val: {len(src_va)}')
print(f'Sample encoded: {src_enc[0]}')
print(f'Decoded: {" ".join(src_i2w[i] for i in src_enc[0] if i != PAD)}')


---
## Part 2 — `BahdanauAttention` Layer

The attention layer computes, at each decoder step:

```
score_i = V · tanh( W1 · enc_out_i + W2 · dec_h )    # (batch, src_len, 1)
alpha   = softmax(score, axis=1)                       # (batch, src_len, 1), sums to 1
context = Σ_i alpha_i · enc_out_i                     # (batch, enc_units)
```

> **Keras 3 note:** use `keras.ops` for all tensor operations — not `tf.*`.
> `ops.expand_dims`, `ops.tanh`, `ops.softmax`, `ops.sum`, `ops.concatenate`
> are backend-agnostic and work in Keras 3 subclassed models.


In [ ]:
class BahdanauAttention(Layer):
    """
    Bahdanau (additive) attention layer.

    Parameters
    ----------
    units : int
        Attention hidden dimension.

    Call inputs
    -----------
    enc_out    : (batch, src_len, enc_units)
    dec_hidden : (batch, dec_units)

    Returns
    -------
    context : (batch, enc_units)      weighted sum of encoder states
    alpha   : (batch, src_len, 1)     attention weights

    Notes
    -----
    Use ops.expand_dims(dec_hidden, 1) to broadcast dec_hidden
    over the src_len dimension before adding to W1(enc_out).
    softmax must be over axis=1 (the src_len positions).
    """

    def __init__(self, units: int, **kwargs):
        super().__init__(**kwargs)
        # YOUR CODE HERE
        raise NotImplementedError

    def call(self, enc_out, dec_hidden):
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
attn = BahdanauAttention(ATTN_DIM)
dummy_enc = np.zeros((2, SRC_MAX, LSTM_UNITS), dtype=np.float32)
dummy_dec = np.zeros((2, LSTM_UNITS), dtype=np.float32)
ctx, alp = attn(dummy_enc, dummy_dec)
print(f'context shape : {ctx.shape}   expected (2, {LSTM_UNITS})')
print(f'alpha shape   : {alp.shape}  expected (2, {SRC_MAX}, 1)')
alpha_sum = np.sum(alp.numpy(), axis=1).flatten()
print(f'alpha row sum : {alpha_sum[:2]}  (should be ~1.0)')
assert tuple(ctx.shape) == (2, LSTM_UNITS)
assert tuple(alp.shape) == (2, SRC_MAX, 1)
print('Shape assertions passed.')


---
## Part 3 — `Seq2SeqAttention` Model

**Architecture overview:**

```
Encoder:
  Embedding(src_vocab, emb_dim)
  Bidirectional LSTM(lstm_units//2) → enc_out (batch, src_len, lstm_units)
  Linear projection → same lstm_units dimension
  Mean pool → init_h, init_c for decoder

Decoder (teacher forcing loop over T_y steps):
  Embedding(tgt_vocab, emb_dim)
  At step t:
    context, alpha = BahdanauAttention(enc_out, h)
    LSTM input      = [tgt_emb_t ; context]
    _, h, c         = LSTM(input, state=[h, c])
    logit_t         = Dense(softmax)( tanh([h ; context]) )
  Output: stack(logits) → (batch, T_y, tgt_vocab)
```

**Why bidirectional encoder?**
With a unidirectional encoder, `enc_out[:, i, :]` at each position `i` accumulates
information from tokens 1 through `i`. The final state `enc_out[:, T, :]` therefore
contains the full sentence. The attention mechanism tends to peak on this final state,
making the attention map flat. With a bidirectional encoder, each position `i` encodes
only *local* context (forward up to `i`, backward from `T` down to `i`),
forcing the attention to distinguish between positions.


In [ ]:
class Seq2SeqAttention(keras.Model):
    """
    Seq2seq with Bahdanau attention and bidirectional encoder.

    Parameters
    ----------
    src_vocab, tgt_vocab : int
    emb_dim : int
    lstm_units : int
        Decoder LSTM hidden size. Encoder uses lstm_units//2 per direction.
    attn_dim : int

    Call signature
    --------------
    inputs = (src_input, tgt_input)
    Returns (batch, T_y, tgt_vocab) — logits at each decoder step.

    Notes
    -----
    Bidirectional LSTM returns: (seq, fwd_h, fwd_c, bwd_h, bwd_c).
    Project the bidirectional output (2*lstm_units//2 = lstm_units per pos)
    through a Dense(lstm_units, tanh) to keep the attention dimension consistent.
    Initialise decoder (h, c) from Dense(tanh)(mean(enc_out)) — NOT from enc_h
    (using enc_h directly causes attention collapse to the final encoder position).
    """

    def __init__(self, src_vocab, tgt_vocab, emb_dim, lstm_units, attn_dim, **kwargs):
        super().__init__(**kwargs)
        # YOUR CODE HERE: define all sub-layers
        # Required:
        #   self.enc_emb     : Embedding
        #   self.enc_bilstm  : Bidirectional(LSTM(lstm_units//2, ...))
        #   self.enc_proj    : Dense(lstm_units, tanh)   # project BiLSTM output
        #   self.init_h      : Dense(lstm_units, tanh)   # decoder h0 from mean
        #   self.init_c      : Dense(lstm_units, tanh)   # decoder c0 from mean
        #   self.attention   : BahdanauAttention(attn_dim)
        #   self.dec_emb     : Embedding
        #   self.dec_lstm    : LSTM(lstm_units, return_state=True)
        #   self.out_proj    : Dense(lstm_units, tanh)   # combines h + context
        #   self.dec_dense   : Dense(tgt_vocab, softmax)
        raise NotImplementedError

    def call(self, inputs, training=False):
        src_input, tgt_input = inputs
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
model = Seq2SeqAttention(SRC_VOCAB, TGT_VOCAB, EMB_DIM, LSTM_UNITS, ATTN_DIM)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Quick shape test
_out = model([src_enc[:2], tgt_enc[:2, :-1]])
print(f'Output shape: {_out.shape}   expected (2, {TGT_MAX-1}, {TGT_VOCAB})')
assert _out.shape == (2, TGT_MAX-1, TGT_VOCAB)
print('Shape OK. Model ready.')
model.summary()


---
## Part 4 — Train

Decoder input: target shifted right (starts with `<SOS>`, drops last token).  
Decoder target: target shifted left (drops `<SOS>`, ends with `<EOS>`).


In [ ]:
history = model.fit(
    [src_tr, tgt_tr[:, :-1]], tgt_tr[:, 1:],
    validation_data=([src_va, tgt_va[:, :-1]], tgt_va[:, 1:]),
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, m in zip(axes, ['loss', 'accuracy']):
    ax.plot(history.history[m],         color='#56c8ba', label='train')
    ax.plot(history.history[f'val_{m}'], color='#e8c47a', label='val')
    ax.set_title(m, color='#e6edf3'); ax.legend()
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117'); plt.tight_layout(); plt.show()
print(f'Final — train acc: {history.history["accuracy"][-1]:.3f}  '
      f'val acc: {history.history["val_accuracy"][-1]:.3f}')


---
## Part 5 — Greedy Decoding

At inference: feed `<SOS>`, run attention + decoder LSTM, take argmax, repeat.
Stop at `<EOS>`. Collect attention weights `alpha` at each step.


In [ ]:
def translate(
    sentence: str,
    model: keras.Model,
    src_w2i: dict,
    tgt_i2w: dict,
    src_max: int,
    tgt_max: int,
) -> tuple[str, np.ndarray]:
    """
    Translate a sentence using greedy decoding.

    Steps:
        1. Encode source → enc_out (bidirectional)
        2. Compute decoder initial state from mean(enc_out)
        3. Loop: attention → LSTM → argmax; stop at EOS
        4. Collect alpha weights

    Parameters
    ----------
    sentence : str
    model    : Seq2SeqAttention
    src_w2i, tgt_i2w : dicts
    src_max, tgt_max : int

    Returns
    -------
    translation : str
    attention   : np.ndarray, shape (output_len, src_max)

    Notes
    -----
    Access model sub-layers directly:
        model.enc_emb, model.enc_bilstm, model.enc_proj
        model.init_h, model.init_c
        model.attention, model.dec_emb, model.dec_lstm
        model.out_proj, model.dec_dense
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Test on sentences NOT in the training set
test_sents = [
    'the black cat sleeps',
    'the red bird flies',
    'the white dog eats',
    'the blue fish swims',
    'the small rabbit jumps',
]
print(f'{"English":<30} French translation')
print('-' * 60)
for sent in test_sents:
    tr, _ = translate(sent, model, src_w2i, tgt_i2w, SRC_MAX, TGT_MAX)
    print(f'{sent:<30} {tr}')


---
## Part 6 — Attention Maps

An attention map has:
- Rows = decoder steps (FR target tokens)
- Columns = encoder steps (EN source tokens)
- Cell (j, i) = α_{t=j, i}: how much the decoder attended to EN position i when generating FR token j

**What to look for:**
- `oiseau` (bird) should attend to `bird` → diagonal alignment
- `noir` (adj) should attend to `black` (adj), NOT to `cat` → off-diagonal
- This off-diagonal pattern is the adjective reordering signature


In [ ]:
def plot_attention_map(
    sentence: str,
    translation: str,
    attention: np.ndarray,
    figsize: tuple = (7, 5),
) -> plt.Figure:
    """
    Plot attention weights as a labelled heatmap.

    Parameters
    ----------
    sentence    : str  (source English)
    translation : str  (predicted French)
    attention   : np.ndarray, shape (tgt_len, src_len)
    figsize     : tuple

    Returns
    -------
    plt.Figure

    Notes
    -----
    Trim attention to actual token lengths (no padding).
    Use cmap='YlOrRd'. Add colorbar.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
for sent in test_sents[:3]:
    tr, attn = translate(sent, model, src_w2i, tgt_i2w, SRC_MAX, TGT_MAX)
    print(f'EN: {sent}  →  FR: {tr}')
    fig = plot_attention_map(sent, tr, attn)
    plt.show()
    print()


**📝 Your analysis**

1. For 'the black cat sleeps': which FR token attends most to 'black'?
   Which attends most to 'cat'? Is there an off-diagonal pattern?
2. For 'the red bird flies': does 'rouge' (red adj) attend to 'red' or to 'bird'?
3. Compare the maps for two sentences with different adjectives.
   Is the alignment pattern consistent?

*Write your observations here.*


---
## Part 7 (Bonus) — Beam Search


In [ ]:
def beam_search(
    sentence: str,
    model: keras.Model,
    src_w2i: dict,
    tgt_i2w: dict,
    src_max: int,
    tgt_max: int,
    beam_width: int = 3,
) -> list[tuple[str, float]]:
    """
    Beam search decoding.
    Returns [(translation, log_score)] sorted by score descending.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
for sent in test_sents[:3]:
    results = beam_search(sent, model, src_w2i, tgt_i2w, SRC_MAX, TGT_MAX, 3)
    greedy, _ = translate(sent, model, src_w2i, tgt_i2w, SRC_MAX, TGT_MAX)
    print(f'Source : {sent}')
    print(f'Greedy : {greedy}')
    for tr, sc in results: print(f'  [{sc:.3f}] {tr}')
    print()


---
## Summary


In [ ]:
print('TP-06 — Seq2Seq + Bahdanau Attention')
print(f'Corpus  : {len(CORPUS)} EN→FR pairs (systematic adjective-order corpus)')
print(f'Vocab   : src={SRC_VOCAB}  tgt={TGT_VOCAB}')
print(f'Model   : EMB={EMB_DIM}  LSTM={LSTM_UNITS}  ATTN={ATTN_DIM}  BiLSTM encoder')
print()
for sent in test_sents:
    tr, _ = translate(sent, model, src_w2i, tgt_i2w, SRC_MAX, TGT_MAX)
    print(f'  {sent} → {tr}')
print()
print('Next: S07 — Self-attention, Transformers, DistilBERT fine-tuning')
